# Length generalization in a small arithmetic transformer

A ~10M-parameter decoder-only transformer is trained on 3-digit addition only, then evaluated on operands with 1–6 digits. We compare six variants:

| Variant | Positional encoding | Answer order | Training data |
|---|---|---|---|
| `baseline` | learned | natural (`579`) | 3-digit |
| `reversed` | learned | reversed (`975`) | 3-digit |
| `nope`     | none    | reversed | 3-digit |
| `rope`     | rotary  | reversed | 3-digit |
| `abacus`   | digit-position only | reversed | 3-digit |
| `abacus_curriculum` | digit-position only | reversed | **mixed 1..5-digit** |

The hypothesis: **learned positional embeddings can't extrapolate** — positions never seen at training time stay at random init, so any sequence longer than the training distribution fails catastrophically. RoPE and NoPE extrapolate further by construction; Abacus encodes *place value* rather than absolute position, so a digit knows it is "the tens-place digit" regardless of where in the sequence it sits. The curriculum variant additionally lets every place-value embedding see gradient by training on mixed digit counts.

**Runtime on a Colab T4:** ~25–35 minutes for the full six-variant sweep. To save time, comment out `abacus_curriculum` (which trains on 1M mixed-digit samples).

## 1. Setup

Clones the repo (if running on Colab), installs dependencies, and verifies GPU availability.

In [ ]:
import os, sys, subprocess, pathlib

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB and not pathlib.Path("jax-transformer").exists():
    subprocess.run(["git", "clone", "--quiet",
                    "https://github.com/sananthanarayan/jax-transformer.git"],
                   check=True)

REPO_ROOT = pathlib.Path("jax-transformer" if IN_COLAB else "..").resolve()
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT / "src"))

if IN_COLAB:
    subprocess.run(["pip", "install", "-q", "-U", "flax>=0.10.0", "optax>=0.2.3"], check=True)

import jax
print("jax devices:", jax.devices())

## 2. Smoke test

Builds the model, runs one forward pass, and prints the untrained loss. Should match `ln(vocab_size) ≈ 2.71`.

In [ ]:
import jax.numpy as jnp
import numpy as np
from flax import nnx

from addition_transformer.data import build_arrays, generate_pairs, max_len_for
from addition_transformer.model import Transformer, TransformerConfig, count_params
from addition_transformer.vocab import VOCAB_SIZE

cfg = TransformerConfig()
model = Transformer(cfg, rngs=nnx.Rngs(0))
print(f"params: {count_params(model)/1e6:.2f}M  pos_encoding={cfg.pos_encoding}  max_len={cfg.max_len}")

pairs = generate_pairs("addition", max_digits=3)[:8]
inp, tgt, mask, dp = build_arrays(pairs, "addition", max_len=cfg.max_len)
logits = model(jnp.asarray(inp))
log_probs = jax.nn.log_softmax(logits, axis=-1)
tgt_lp = jnp.take_along_axis(log_probs, jnp.asarray(tgt)[..., None], axis=-1).squeeze(-1)
loss = (-tgt_lp * jnp.asarray(mask)).sum() / max(int(jnp.asarray(mask).sum()), 1)
print(f"untrained masked loss = {float(loss):.4f}  (random baseline = {np.log(VOCAB_SIZE):.4f})")

## 3. Train the variants

Each variant trains on 3-digit addition only (except `abacus_curriculum`, which trains on mixed 1..5-digit). We use **4 epochs** to keep the Colab session short; bump `EPOCHS` up if you have time.

In [ ]:
import time
from addition_transformer.train import train_model
from addition_transformer.eval import eval_at_digits, per_digit_position_accuracy

OP = "addition"
TRAIN_DIGITS = 3
EVAL_DIGITS = list(range(1, 7))
MODEL_MAX_LEN = max_len_for(OP, max(EVAL_DIGITS))
EPOCHS = 4
EVAL_SAMPLES = 300
SEED = 0

VARIANTS = [
    {"name": "baseline", "label": "Learned PE + natural order",
     "pos_encoding": "learned", "reverse_answer": False, "mixed_digits": False},
    {"name": "reversed", "label": "Learned PE + reversed answers",
     "pos_encoding": "learned", "reverse_answer": True,  "mixed_digits": False},
    {"name": "nope",     "label": "NoPE + reversed answers",
     "pos_encoding": "none",    "reverse_answer": True,  "mixed_digits": False},
    {"name": "rope",     "label": "RoPE + reversed answers",
     "pos_encoding": "rope",    "reverse_answer": True,  "mixed_digits": False},
    {"name": "abacus",   "label": "Abacus (clean) + reversed answers",
     "pos_encoding": "abacus",  "reverse_answer": True,  "mixed_digits": False},
    # Comment this out to save ~5 min on a T4:
    {"name": "abacus_curriculum", "label": "Abacus + length curriculum (1..5 digits)",
     "pos_encoding": "abacus",  "reverse_answer": True,  "mixed_digits": True},
]

results = {}
for v in VARIANTS:
    print(f"\n=== {v['name']}: {v['label']} ===")
    t0 = time.time()
    max_digits = (max(EVAL_DIGITS) - 1) if v["mixed_digits"] else TRAIN_DIGITS
    model = train_model(
        op=OP,
        max_digits=max_digits,
        epochs=EPOCHS,
        seed=SEED,
        reverse_answer=v["reverse_answer"],
        pos_encoding=v["pos_encoding"],
        model_max_len=MODEL_MAX_LEN,
        eval_samples=600,
        mixed_digits=v["mixed_digits"],
        mixed_min_digits=1,
        mixed_n_samples=500_000 if v["mixed_digits"] else 1_000_000,
        log_prefix=f"[{v['name']}] ",
    )
    accs, per_pos = {}, {}
    for d in EVAL_DIGITS:
        acc = eval_at_digits(model, OP, d, n_samples=EVAL_SAMPLES,
                             reverse_answer=v["reverse_answer"], seed=SEED + 1000)
        accs[d] = acc
        pp, _ = per_digit_position_accuracy(model, OP, d, n_samples=EVAL_SAMPLES,
                                            reverse_answer=v["reverse_answer"], seed=SEED + 1000)
        per_pos[d] = [None if (x != x) else float(x) for x in pp]
        print(f"[{v['name']}] digits={d}: {acc*100:.2f}%")
    results[v["name"]] = {"label": v["label"], "accuracies": accs,
                          "per_position": per_pos,
                          "train_time_sec": time.time() - t0}

## 4. The headline chart

In [ ]:
import matplotlib.pyplot as plt

COLORS = {
    "baseline":          "#d62728",
    "reversed":          "#ff7f0e",
    "nope":              "#2ca02c",
    "rope":              "#1f77b4",
    "abacus":            "#9467bd",
    "abacus_curriculum": "#17becf",
}

fig, ax = plt.subplots(figsize=(8.0, 4.8))
ax.axvspan(EVAL_DIGITS[0] - 0.1, TRAIN_DIGITS + 0.1, alpha=0.08, color="gray",
           label=f"Training distribution (≤{TRAIN_DIGITS} digits)")
for name, r in results.items():
    xs = sorted(r["accuracies"].keys())
    ys = [r["accuracies"][d] * 100 for d in xs]
    ax.plot(xs, ys, marker="o", linewidth=2, markersize=7,
            color=COLORS.get(name, "tab:gray"), label=r["label"])
ax.set_xlabel("Operand digit count")
ax.set_ylabel("Exact-match accuracy (%)")
ax.set_title(f"Length generalization on {OP} (trained on ≤{TRAIN_DIGITS} digits)")
ax.set_xticks(EVAL_DIGITS)
ax.set_ylim(-2, 102)
ax.grid(True, alpha=0.3)
ax.legend(loc="upper right", framealpha=0.95, fontsize=8.5)
fig.tight_layout()

pathlib.Path("results").mkdir(exist_ok=True)
fig.savefig("results/length_gen.png", dpi=150)
plt.show()

## 5. Per-digit-position heatmap

Each panel is one variant. Rows = operand digit count; columns = digit position within the answer (0 = ones, increasing toward the MSB). Color = error rate.

In [ ]:
import math

variants = list(results.items())
max_pos = max(len(r["per_position"][d]) for _, r in variants for d in EVAL_DIGITS)
cols = 3
rows = math.ceil(len(variants) / cols)
fig, axes = plt.subplots(rows, cols, figsize=(4.0 * cols, 2.6 * rows), squeeze=False)

for i, (name, r) in enumerate(variants):
    ax = axes[i // cols][i % cols]
    grid = np.full((len(EVAL_DIGITS), max_pos), np.nan, dtype=np.float32)
    for ri, d in enumerate(EVAL_DIGITS):
        row = r["per_position"].get(d) or []
        for ci, val in enumerate(row):
            grid[ri, ci] = np.nan if val is None else (1.0 - float(val))
    im = ax.imshow(grid, aspect="auto", origin="lower", cmap="Reds",
                   vmin=0.0, vmax=1.0, interpolation="nearest")
    ax.set_title(r["label"], fontsize=9)
    ax.set_xlabel("Answer digit position (0 = ones)", fontsize=8)
    ax.set_ylabel("Operand digit count", fontsize=8)
    ax.set_yticks(range(len(EVAL_DIGITS)))
    ax.set_yticklabels(EVAL_DIGITS)
    ax.set_xticks(range(max_pos))
    ax.set_xticklabels(range(max_pos))
    ax.tick_params(labelsize=8)
for j in range(len(variants), rows * cols):
    axes[j // cols][j % cols].axis("off")
fig.suptitle(f"Per-digit-position error rate ({OP}, trained on ≤{TRAIN_DIGITS} digits)",
             fontsize=11)
fig.colorbar(im, ax=axes, shrink=0.6, pad=0.02).set_label("Error rate", fontsize=9)
fig.savefig("results/per_digit_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()